In [ ]:
from pathlib import Path
from typing import List, Dict, Tuple, Union
from collections import defaultdict
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm.auto import tqdm
from PIL import Image
import h5py
import scipy.io
from scipy.stats import zscore

import nilearn as nl
import nilearn.image as nl_image
import nilearn.plotting as nl_plotting
import nibabel as nib

In [ ]:
SUBJECTS = [
    f"subj{i:02d}" for i in range(1, 9)
]
data_space = "nativesurface"
beta_type ="betas_fithrf_GLMdenoise_RR"

# SUBJECTS

In [ ]:
ds_dir = '${MBS_NSD_DIR}'
# ds_path = '${MBS_NSD_DIR}/nsddata_betas/ppdata/subj01/fsaverage/betas_fithrf_GLMdenoise_RR'

ds_dir = Path(ds_dir)

# list((ds_path ).iterdir())

# Noise Ceiling

In [ ]:
ncsnr2nc = lambda x: 100 * (x**2) / (x**2 + 1/3)

In [ ]:
noise_ceiling_masks = {}

for sub in tqdm(SUBJECTS):
    nc_lh_filepath = ds_dir / "nsddata_betas/ppdata" / sub / data_space / beta_type / "lh.ncsnr.mgh"
    nc_rh_filepath = ds_dir / "nsddata_betas/ppdata" / sub / data_space / beta_type / "rh.ncsnr.mgh"

    nc_lh = nib.load(nc_lh_filepath).get_fdata()
    nc_rh = nib.load(nc_rh_filepath).get_fdata()
    
    nc = np.concatenate([np.squeeze(nc_lh), np.squeeze(nc_rh)], axis=0)
    
    noise_ceiling_masks[sub] = ncsnr2nc(nc)

    

In [ ]:
ncsnr2nc(0.2)

In [ ]:
thresh = 10

for sub, mask in noise_ceiling_masks.items():
    print(f"Subject: {sub}, available voxels: {(mask > thresh).sum()}")



# ROI masks

In [ ]:
def load_ctab(file_path):
    ctab = pd.read_csv(
        file_path,
        sep="\s+",   # split on any whitespace
        header=None,
        comment='#',             # just in case there are comments
        # skiprows=1,               # skip the "num entries" line,
        names=['roi_id', 'roi_name']
    )
    return ctab


In [ ]:
ROI_files = [
    "streams", 
    "prf-visualrois",
    "nsdgeneral",
    "floc-words",
    # "floc-places",
    # "floc-faces",
    # "floc-bodies"
]

In [ ]:
ROI_raw_masks = defaultdict(dict)
ROI_metadata = defaultdict(dict)


all_rois = []
for sub in tqdm(SUBJECTS):
    for meta_roi in tqdm(ROI_files, leave=False):
        roi_meta_mask_left = ds_dir / "nsddata/freesurfer" / sub / "label" / f"lh.{meta_roi}.mgz"
        roi_meta_mask_right = ds_dir / "nsddata/freesurfer" / sub / "label" / f"rh.{meta_roi}.mgz"

        roi_meta_mask_left = np.squeeze(nib.load(roi_meta_mask_left).get_fdata())
        roi_meta_mask_right = np.squeeze(nib.load(roi_meta_mask_right).get_fdata())
        roi_meta_mask = np.concatenate([roi_meta_mask_left, roi_meta_mask_right], axis=0)

        metadata = load_ctab(ds_dir / "nsddata/freesurfer" / sub / "label" / f"{meta_roi}.mgz.ctab")
        ROI_metadata[sub][meta_roi] = metadata
        
        for row_id, row in metadata[1:].iterrows():
            roi_name = row.roi_name
            roi_id = row.roi_id
            
            roi_mask = roi_meta_mask==roi_id
            
            ROI_raw_masks[sub][roi_name] = roi_mask
            all_rois.append(roi_name)


    # # Whole brain mask
    # ROI_raw_masks[sub]["whole_brain"] = np.ones_like(roi_meta_mask, dtype=bool)
    # all_rois.append("whole_brain")


all_rois = sorted(set(all_rois))
    


In [ ]:
all_rois

In [ ]:
ROI_mapping = {
    "V1": ["V1d", "V1v"],
    "V2": ["V2d", "V2v"],
    # "V3": ["V3d", "V3v"],
    "V4": ["hV4"],
    "IT": [
        "midlateral",
        "midparietal",
        "midventral",
        "parietal",
        "lateral",
        "ventral"
    ],
    "VWFA": ["OWFA", "VWFA-1", "VWFA-2"],
    "vision": ["nsdgeneral"]
}

In [ ]:
ROI_masks = defaultdict(dict)

for sub in tqdm(SUBJECTS):
    for roi_name, roi_list in ROI_mapping.items():
        roi_mask = np.zeros_like(ROI_raw_masks[sub]["nsdgeneral"], dtype=bool)
        
        for roi_ in roi_list:
        
            roi_mask |= ROI_raw_masks[sub][roi_]
            
        #Apply nsdgeneral mask
        roi_mask &= ROI_raw_masks[sub]["nsdgeneral"]
        
        ROI_masks[sub][roi_name] = roi_mask
        
    # Add whole_brain
    roi_mask = np.ones_like(ROI_raw_masks[sub]["nsdgeneral"], dtype=bool)
    ROI_masks[sub]["whole_brain"] = roi_mask

In [ ]:
for sub in tqdm(SUBJECTS):
    print(sub)
    for roi_name, roi_mask in ROI_masks[sub].items():
        print("\t", roi_name, roi_mask.sum())

# Noise Ceiled ROI Masks

In [ ]:
NOISE_CEILING_THRESHOLD = 10

In [ ]:
ROI_masks_noise_ceiled = defaultdict(dict)

for sub in tqdm(SUBJECTS):
    for roi_name, roi_mask in ROI_masks[sub].items():
        nc_mask = noise_ceiling_masks[sub] >= NOISE_CEILING_THRESHOLD
        ROI_masks_noise_ceiled[sub][roi_name] = roi_mask & nc_mask

In [ ]:
for sub in tqdm(SUBJECTS):
    print(sub)
    for roi_name, roi_mask in ROI_masks_noise_ceiled[sub].items():
        print("\t", roi_name, roi_mask.sum())

# Process data

In [ ]:
subject_data = defaultdict(dict)

for sub in tqdm(SUBJECTS):
    sub_data_dir = ds_dir / "nsddata_betas/ppdata" / sub / data_space / beta_type
    
    subj_data = []
    surface_data_lh = sorted(list(sub_data_dir.glob("lh.betas_session*.hdf5")))
    surface_data_rh = sorted(list(sub_data_dir.glob("rh.betas_session*.hdf5")))

    for lh_path, rh_path in tqdm(zip(surface_data_lh, surface_data_rh), total=len(surface_data_lh), leave=False):
        lh_surface = h5py.File(lh_path, 'r')
        rh_surface = h5py.File(rh_path, 'r')

        lh_surface_array = lh_surface['betas'][:]  # 750 x N1
        rh_surface_array = rh_surface['betas'][:]  # 750 x N2

        surface_data = np.concatenate([lh_surface_array, rh_surface_array], axis=1) # 750 x (N1+N2)
        surface_data = surface_data.astype(np.float16) / 300
        surface_data = zscore(surface_data, axis=0, ddof=1)
        
        subj_data.append(surface_data)
        
        break
        
    subject_data[sub] = np.concatenate(subj_data, axis=0)

# Save

In [ ]:
save_dir = "${MBS_DATA_PREP_OUTPUT_DIR}"
save_dir = Path(save_dir)

save_dir = save_dir / f"nsd_{data_space}2.hdf5"

if not save_dir.parent.exists():
    save_dir.parent.mkdir(parents=True, exist_ok=False)

In [ ]:
with h5py.File(save_dir, 'w') as f:
    f.attrs['subjects'] = SUBJECTS
    f.attrs['data_space'] = data_space
    f.attrs['beta_type'] = beta_type
    f.attrs['rois'] = list(ROI_mapping.keys())
    f.attrs['ROI_mapping'] = json.dumps(ROI_mapping)
    
    # Create noise ceiling dataset
    # for sub in tqdm(SUBJECTS, desc="Creating noise ceiling masks"):
    #     f.create_dataset(f"noise_ceiling_masks/{sub}", data=subject_data[sub])
        
    # # Create ROI masks
    # for sub in tqdm(SUBJECTS, desc="Creating ROI masks"):
    #     for roi_name, roi_mask in ROI_masks[sub].items():
    #         f.create_dataset(f"roi_masks/{sub}/{roi_name}", data=roi_mask)
            
    # # Create noise ceiled ROI masks
    # for sub in tqdm(SUBJECTS, desc="Creating noise ceiled ROI masks"):
    #     for roi_name, roi_mask in ROI_masks_noise_ceiled[sub].items():
    #         f.create_dataset(f"roi_masks_noise_ceiling/{sub}/{roi_name}", data=roi_mask)

    # Write data
    for sub in tqdm(SUBJECTS, desc="Writing data"):
        for roi_name, roi_mask in ROI_masks_noise_ceiled[sub].items():
            # if roi_name=="whole_brain":continue
            f.create_dataset(f"data/{sub}/{roi_name}", data=subject_data[sub][:, roi_mask])

# Test

In [ ]:
data_path = "${MBS_DATA_PREP_OUTPUT_DIR}"
data_path = Path(data_path)
data_path = data_path / f"nsd_{data_space}2.hdf5"
assert data_path.exists(), "Data path does not exist"

In [ ]:
with h5py.File(data_path, 'r') as f:
    subjects = f.attrs['subjects']
    rois = f.attrs['rois']
    
    for sub in tqdm(subjects):
        print(f"Subject: {sub}")
        for roi in rois:
            print(f"\t ROI: {roi}", f[f"data/{sub}/{roi}"].shape)
    

In [ ]:
ff = h5py.File(data_path, 'r')
subjects = ff.attrs['subjects']
rois = ff.attrs['rois']
# ff['data/subj01/IT'].keys()
rois
ff.close()

# for sub in tqdm(subjects):
#     for roi in rois:
#         print(f"Subject: {sub}, ROI: {roi}", ff[f"data/{sub}/{roi}"].shape)
